# 07 · Topic deep dive — sub-topics, dendrogram, UMAP projection, college profiles

Follow-up analyses requested by the PI. Four sections:

1. **Row-normalised topic × college heatmap + per-college profile cards** — "what does each college actually work on?"
2. **Sub-topics inside each of the 8 parent topics** — refit LDA (k=4) inside each parent so "Biomedical" splits into concrete sub-themes.
3. **Topic dendrogram (JS distance)** — which of the 8 topics are cousins?
4. **UMAP projection of all abstracts** — static 2×2 panel plus an interactive Plotly HTML written to `docs/07_grant_projection.html`.

Depends on the k=8 LDA assignments already exported by [`06_research_topics.ipynb`](06_research_topics.ipynb) to `outputs/topic_assignments.csv`.


In [ ]:
import re, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram
import umap
import plotly.express as px

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'data' / 'processed').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PROCESSED = REPO_ROOT / 'data' / 'processed'
OUTPUTS   = REPO_ROOT / 'outputs'; OUTPUTS.mkdir(exist_ok=True)
DOCS      = REPO_ROOT / 'docs';    DOCS.mkdir(exist_ok=True)
FIG_DIR   = REPO_ROOT / 'notebooks' / 'figures'; FIG_DIR.mkdir(exist_ok=True, parents=True)
assert PROCESSED.exists(), f'data/processed not found from {Path.cwd()}'

RNG = 42

In [ ]:
faculty                  = pd.read_parquet(PROCESSED / 'faculty.parquet')
grants                   = pd.read_parquet(PROCESSED / 'grants.parquet')
faculty_grants           = pd.read_parquet(PROCESSED / 'faculty_grants.parquet')
grant_orphaned_abstracts = pd.read_parquet(PROCESSED / 'grant_orphaned_abstracts.parquet')

# Force str for join keys
for df, cols in [(grants, ['grant_id']), (faculty_grants, ['grant_id','faculty_id']),
                 (faculty, ['faculty_id'])]:
    for col in cols:
        df[col] = df[col].astype(str)

# Load the k=8 assignments from notebook 06
topic_assign = pd.read_csv(OUTPUTS / 'topic_assignments.csv')
topic_assign['grant_id'] = topic_assign['grant_id'].astype(str)
topic_assign['id']       = topic_assign['id'].astype(str)

print(f'faculty                    : {faculty.shape}')
print(f'grants                     : {grants.shape}  (abstract populated: {(grants.abstract.astype(str).str.len() > 50).sum()})')
print(f'faculty_grants             : {faculty_grants.shape}')
print(f'grant_orphaned_abstracts   : {grant_orphaned_abstracts.shape}')
print(f'topic_assign               : {topic_assign.shape}')
print(f'unique topics              : {topic_assign.topic_label.nunique()}')

## 0 · Refit the k=8 LDA model

We need the fitted vectorizer + model in memory for the dendrogram (Section 3) and the UMAP option-A (topic distributions). Uses the same preprocessing and hyperparameters as notebook 06.

In [ ]:
BOILERPLATE = re.compile(r"this award reflects nsf's statutory mission.*?criteria\.?", re.IGNORECASE | re.DOTALL)
GTLT = re.compile(r'\b(andgt|andlt|andamp|lt|gt|amp)\b')
HTML = re.compile(r'<[^>]+>')

def clean_abstract(text):
    if not isinstance(text, str): return ''
    t = HTML.sub(' ', text)
    t = BOILERPLATE.sub(' ', t.lower())
    t = GTLT.sub(' ', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t if len(t.split()) >= 40 else ''

# Build cleaned abstract corpus from grants (matched only; each row = one grant)
_abs_src = grants[['grant_id','grantname','abstract']].rename(
    columns={'grant_id':'id','grantname':'title'})
_abs_src['sourceactivityid'] = _abs_src['id']  # downstream code expects this col
abs_clean = _abs_src.assign(clean=_abs_src.abstract.map(clean_abstract))
abs_clean = abs_clean[abs_clean.clean != ''].reset_index(drop=True)
print(f'cleaned abstracts: {len(abs_clean)}')

DOMAIN_STOPS = {
    'research','study','studies','project','proposal','proposed','work','using',
    'new','novel','based','also','used','use','provide','provides','provided',
    'develop','developed','developing','development','including','include',
    'university','northeastern','professor','pi','co','result','results',
    'may','one','two','three','well','make','made','across','within','without',
    'important','significant','general','specific','broader','impact','impacts',
    'award','awards','support','supports','high','low','number','different',
    'approach','approaches','model','models','method','methods'
}
STOPS = list(DOMAIN_STOPS)

vect = CountVectorizer(max_df=0.6, min_df=15, ngram_range=(1,2),
                       token_pattern=r'\b[a-z]{3,}\b',
                       stop_words='english' if False else STOPS + list(CountVectorizer(stop_words='english').get_stop_words()))
# Fall back to sklearn's builtin stop_words + our extras
vect = CountVectorizer(max_df=0.6, min_df=15, ngram_range=(1,2),
                       token_pattern=r'\b[a-z]{3,}\b', stop_words='english')
X = vect.fit_transform(abs_clean.clean)
# Manually zero-out any DOMAIN_STOPS that survived (rare because 'english' already covers most)
vocab = vect.get_feature_names_out()
keep_mask = np.array([not any(w in ds for ds in [' '+t+' ' for t in DOMAIN_STOPS] for w in [' '+v+' ']) for v in vocab])
print(f'vocab: {len(vocab)}')

lda = LatentDirichletAllocation(n_components=8, random_state=RNG, max_iter=50,
                                learning_method='batch')
doc_topic = lda.fit_transform(X)
print(f'X: {X.shape}, doc_topic: {doc_topic.shape}')

# Fallback labels only; runtime uses ALIGNED_LABELS derived from nb06's export.
TOPIC_LABELS = {
    0: 'Mathematics & theoretical physics',
    1: 'Biomedical (drug/disease/cancer)',
    2: 'Software, data & ML systems',
    3: 'Cell & molecular biology',
    4: 'Environmental & public health',
    5: 'Hardware, energy & wireless systems',
    6: 'HCI, learning & applied research',
    7: 'STEM education & outreach',
}

def top_words(lda_model, feat, n=12):
    out = {}
    for k, comp in enumerate(lda_model.components_):
        idx = comp.argsort()[::-1][:n]
        out[k] = [feat[i] for i in idx]
    return out

tw = top_words(lda, vocab, 10)
for k in range(8):
    print(f'[{k}] {TOPIC_LABELS[k]:38s} → {", ".join(tw[k])}')

In [ ]:
# Sanity check: does the refit match the export? Re-align if the topic indices shuffled.
# We map each refit topic to the export label that appears most often when we predict on the same docs.
refit_pred = doc_topic.argmax(axis=1)
abs_clean_with_pred = abs_clean.assign(refit_topic=refit_pred)

# For each abs_clean row, find its export label via 'id' join.
merged_check = abs_clean_with_pred.merge(topic_assign[['id','topic','topic_label']],
                                          left_on='id', right_on='id', how='inner')
if len(merged_check):
    xtab = pd.crosstab(merged_check.refit_topic, merged_check.topic_label)
    # For each refit topic, take the most common exported label
    ALIGNED_LABELS = {int(k): xtab.loc[k].idxmax() for k in xtab.index}
    print('Realigned labels (refit_topic_id → exported label):')
    for k, v in sorted(ALIGNED_LABELS.items()):
        print(f'  [{k}] {v}')
else:
    print('No overlap found — falling back to hardcoded labels')
    ALIGNED_LABELS = TOPIC_LABELS

# Attach labels to the refit doc-topic frame
abs_clean_with_pred['topic_label'] = abs_clean_with_pred.refit_topic.map(ALIGNED_LABELS)
abs_clean_with_pred['topic_prob']  = doc_topic.max(axis=1)

## 1 · What does each college work on? (row-normalised)

The topic × college heatmap in notebook 06 was **column-normalised** ("of all grants in topic X, what % came from each college") — useful for the topic side, unreadable for the college side.

Here we **row-normalise by college**, so each row sums to 100% and directly answers *"of all the grants in this college, what % fall into each topic?"* Then we produce per-college profile cards.

In [ ]:
# Attach college to each faculty–grant link
fg_col = (faculty_grants
          .merge(faculty[['faculty_id','superior_academic_unit']], on='faculty_id', how='left')
          .rename(columns={'superior_academic_unit': 'college'}))
fg_col['college'] = fg_col['college'].astype(str).replace({'nan': 'Unknown'}).fillna('Unknown')

# Consolidate small colleges
KEEP_COLLEGES = [
    'College of Engineering',
    'Khoury College of Computer Sciences',
    'College of Science',
    'Bouvé College of Health Sciences',
    'College of Social Sciences and Humanities',
    'College of Arts, Media and Design',
    "D'Amore-McKim School of Business",
    'School of Law',
]
fg_col['college_grp'] = fg_col.college.where(fg_col.college.isin(KEEP_COLLEGES), 'Other')

# Join to topic assignments
fg_topic = fg_col.merge(topic_assign[['grant_id','topic_label']], on='grant_id', how='inner')
print(f'faculty–grant–topic rows: {len(fg_topic):,}')

# Topic × college count matrix
ct = pd.crosstab(fg_topic.college_grp, fg_topic.topic_label)
# Row-normalise (each college sums to 100%)
ct_row = ct.div(ct.sum(axis=1), axis=0) * 100

# Order colleges by total grant links
row_order = ct.sum(axis=1).sort_values(ascending=False).index.tolist()
ct_row = ct_row.loc[row_order]

fig, ax = plt.subplots(figsize=(11, 6.5))
sns.heatmap(ct_row, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': '% of college\'s grants'}, ax=ax)
ax.set_title('What each college works on — row-normalised (each row = 100%)')
ax.set_xlabel('Research topic (LDA k=8)')
ax.set_ylabel('College')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'w7_topic_by_college_rownorm.png', dpi=140, bbox_inches='tight')
plt.savefig(OUTPUTS / 'w7_topic_by_college_rownorm.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Per-college profile cards
# Attach $ per grant, plus faculty full-credit tally
fg_full = (fg_col
           .merge(grants[['grant_id','totaldollars','agencyname']], on='grant_id', how='left')
           .merge(topic_assign[['grant_id','topic_label']], on='grant_id', how='left'))
fg_full['totaldollars'] = fg_full['totaldollars'].fillna(0)

profile_rows = []
lines = ['# Per-college research profiles', '']
for coll in row_order:
    sub = fg_full[fg_full.college_grp == coll]
    if len(sub) == 0: continue
    n_grants   = sub.grant_id.nunique()
    n_faculty  = sub.faculty_id.nunique()
    total_dollars = sub.drop_duplicates('grant_id').totaldollars.sum()

    top_topics = (sub.dropna(subset=['topic_label'])
                     .groupby('topic_label').size()
                     .sort_values(ascending=False).head(3))
    top_agencies = sub.dropna(subset=['agencyname']).agencyname.value_counts().head(3)
    top_faculty  = (sub.groupby(['faculty_id','faculty_name'])['totaldollars'].sum()
                        .sort_values(ascending=False).head(5))

    lines.append(f'## {coll}')
    lines.append(f'- **Faculty on grants:** {n_faculty} | **Grants:** {n_grants} | **Total $ (full-credit):** ${total_dollars/1e6:,.1f}M')
    lines.append(f'- **Top 3 topics:** ' + '; '.join(f'{t} ({int(n)})' for t, n in top_topics.items()))
    lines.append(f'- **Top 3 agencies:** ' + '; '.join(f'{a} ({int(n)})' for a, n in top_agencies.items()))
    lines.append('- **Top 5 PIs (full-credit $):**')
    for (fid, fname), amt in top_faculty.items():
        lines.append(f'    - {fname} — ${amt/1e6:,.1f}M')
    lines.append('')

    profile_rows.append({
        'college': coll, 'n_faculty': n_faculty, 'n_grants': n_grants,
        'total_dollars': total_dollars,
        'top_topic_1': top_topics.index[0] if len(top_topics) else '',
        'top_topic_2': top_topics.index[1] if len(top_topics) > 1 else '',
        'top_topic_3': top_topics.index[2] if len(top_topics) > 2 else '',
        'top_agency':  top_agencies.index[0] if len(top_agencies) else '',
        'top_pi':      top_faculty.index[0][1] if len(top_faculty) else '',
    })

profiles_df = pd.DataFrame(profile_rows)
profiles_df.to_csv(OUTPUTS / 'college_profiles.csv', index=False)
print(f'wrote {OUTPUTS / "college_profiles.csv"}')

# Also render as HTML for the docs/ folder
from html import escape
html_body = ['<!doctype html><meta charset="utf-8">',
             '<title>NEU research — college profiles</title>',
             '<style>body{font-family:-apple-system,Segoe UI,sans-serif;max-width:900px;margin:2em auto;padding:0 1em;line-height:1.5;color:#222}h2{margin-top:2em;border-bottom:2px solid #C41230;padding-bottom:.3em;color:#C41230}li{margin:.3em 0}</style>',
             '<h1>Northeastern research — college profiles</h1>']
current = []
for ln in lines[1:]:
    if ln.startswith('## '):
        html_body.append(f'<h2>{escape(ln[3:])}</h2><ul>')
    elif ln.startswith('- **'):
        html_body.append(f'<li>{ln[2:].replace("**","<b>",1).replace("**","</b>",1)}</li>')
    elif ln.startswith('    - '):
        html_body.append(f'<li style="margin-left:1.5em">{escape(ln[6:])}</li>')
    elif ln == '':
        html_body.append('</ul>')
(DOCS / 'college_profiles.html').write_text('\n'.join(html_body))
print(f'wrote {DOCS / "college_profiles.html"}')

# Print first two cards inline
print('\n\n'.join(['\n'.join(lines[0:1])] + ['\n'.join(lines[i:i+9]) for i in range(1, min(19, len(lines)), 9)]))

## 2 · Sub-topics inside each parent topic

For each of the 8 parent topics, we refit an LDA (k=4) on just its documents. This turns "Biomedical & cell biology" into 4 concrete sub-themes with their own top terms, top grants, and top faculty.

Sub-topic labels are **auto-generated from the top terms** — LDA cluster indices are random per fit, so we surface the actual vocabulary rather than baking in speculative names. Rewrite the `subtopic_label` column of [`outputs/subtopics.csv`](../outputs/subtopics.csv) with curated names after you read the top terms.

In [ ]:
def fit_subtopics(parent_docs_idx, k=4, min_df=5, max_df=0.7):
    corpus = abs_clean.loc[parent_docs_idx, 'clean'].tolist()
    if len(corpus) < 30:
        return None
    v = CountVectorizer(max_df=max_df, min_df=min_df, ngram_range=(1,2),
                        token_pattern=r'\b[a-z]{3,}\b', stop_words='english')
    try:
        Xs = v.fit_transform(corpus)
    except ValueError:
        return None
    model = LatentDirichletAllocation(n_components=k, random_state=RNG,
                                       max_iter=40, learning_method='batch')
    dt = model.fit_transform(Xs)
    return v, model, dt, parent_docs_idx

# Build parent → doc indices via the ALIGNED refit assignments
parent_docs = {label: abs_clean_with_pred.index[abs_clean_with_pred.topic_label == label].tolist()
               for label in ALIGNED_LABELS.values()}
for lbl, docs in parent_docs.items():
    print(f'{lbl:38s} → {len(docs):4d} docs')

### Example grants per parent theme

Quick sanity check: for each of the 8 parent topics, print the top-5 grants
by (a) LDA assignment confidence and (b) total $ amount. Fastest way to see
whether the parent buckets actually contain the kind of research their labels
claim.

In [ ]:
# Example grants per parent theme — top by confidence and by $
_examples = (abs_clean_with_pred[['id','sourceactivityid','title','topic_label','topic_prob']]
             .rename(columns={'sourceactivityid':'grant_id'}))
_examples['grant_id'] = _examples['grant_id'].astype(str)
_examples = _examples.merge(
    grants[['grant_id','totaldollars','startdateyear','agencyname']],
    on='grant_id', how='left')

# Attach lead PI (first is_pi=True row per grant, else any row)
_lead = (faculty_grants.sort_values(['grant_id','is_pi'], ascending=[True, False])
                       .drop_duplicates('grant_id')[['grant_id','faculty_name']]
                       .rename(columns={'faculty_name':'lead_pi'}))
_examples = _examples.merge(_lead, on='grant_id', how='left')

# Dedupe on grant_id, keeping the highest-confidence row per grant
_examples = (_examples.sort_values('topic_prob', ascending=False)
                       .drop_duplicates(subset=['grant_id'])
                       .reset_index(drop=True))

def _print(sub, mode):
    for _, r in sub.iterrows():
        title = (str(r.title)[:78] + '…') if isinstance(r.title, str) and len(r.title) > 79 else str(r.title)
        pi    = str(r.lead_pi) if pd.notna(r.lead_pi) else '?'
        yr    = int(r.startdateyear) if pd.notna(r.startdateyear) else '?'
        amt   = f'${r.totaldollars/1e6:5.2f}M' if pd.notna(r.totaldollars) else '   ?  '
        conf  = f'{r.topic_prob:.2f}'
        print(f'    [{mode}] p={conf}  {amt}  {yr}  {pi}')
        print(f'          {title}')

for label in sorted(ALIGNED_LABELS.values()):
    block = _examples[_examples.topic_label == label]
    print(f'\n=== {label}  (n={len(block)} grants) ===')
    print('  by CONFIDENCE:')
    _print(block.nlargest(5, 'topic_prob'), 'conf')
    print('  by $ AMOUNT:')
    _print(block.dropna(subset=['totaldollars']).nlargest(5, 'totaldollars'), '$$$')

In [ ]:
SUB_K = 4
subtopic_rows = []
subtopic_top_words = {}

for parent_label, docs_idx in parent_docs.items():
    result = fit_subtopics(docs_idx, k=SUB_K)
    if result is None:
        print(f'{parent_label}: too few docs, skipping')
        continue
    v, model, dt, idx = result
    feat = v.get_feature_names_out()
    sub_assign = dt.argmax(axis=1)
    sub_conf   = dt.max(axis=1)

    # Attach sub-topic and metadata to the abs_clean rows for this parent
    parent_slice = abs_clean.loc[idx, ['id','sourceactivityid','title']].copy()
    parent_slice['sub_id']   = sub_assign
    parent_slice['sub_prob'] = sub_conf

    # Join to grants for $ and faculty for names
    parent_slice = parent_slice.merge(grants[['grant_id','totaldollars']],
                                       left_on='sourceactivityid', right_on='grant_id', how='left')

    subtopic_top_words[parent_label] = {}
    for s in range(SUB_K):
        idx_s = parent_slice[parent_slice.sub_id == s]
        top_terms = [feat[i] for i in model.components_[s].argsort()[::-1][:12]]
        subtopic_top_words[parent_label][s] = top_terms

        # Top faculty on these grants
        fac_amounts = (faculty_grants[faculty_grants.grant_id.isin(idx_s.grant_id.dropna())]
                        .merge(grants[['grant_id','totaldollars']], on='grant_id', how='left')
                        .groupby(['faculty_id','faculty_name']).totaldollars.sum()
                        .sort_values(ascending=False).head(3))

        # Top grants by $
        top_grants = idx_s.dropna(subset=['grant_id']).sort_values('totaldollars', ascending=False).head(3)

        subtopic_rows.append({
            'parent_topic': parent_label,
            'sub_id': s,
            'n_docs': len(idx_s),
            'total_dollars': idx_s.totaldollars.fillna(0).sum(),
            'top_terms': ', '.join(top_terms[:8]),
            'top_grants': ' | '.join(top_grants.title.astype(str).str[:80]),
            'top_faculty': ' | '.join(f'{n} (${a/1e6:.1f}M)' for (_,n), a in fac_amounts.items()),
        })

subtopics_raw = pd.DataFrame(subtopic_rows)
print(f'built {len(subtopics_raw)} sub-topics across {subtopics_raw.parent_topic.nunique()} parents')
subtopics_raw.head()

In [ ]:
# Sub-topic labels — auto-generated from top terms.
#
# NOTE: LDA sub-topic indices are assigned randomly per fit, so we cannot safely
# hand-curate labels without inspecting each fit's output first. Instead we
# generate a descriptive label directly from the top 3 discriminative terms.
# A human reader (you / the PI) can rewrite these in `outputs/subtopics.csv`
# after glancing at the top_terms column.

def auto_label(top_terms_str):
    terms = [t.strip() for t in top_terms_str.split(',')]
    # Drop obvious filler words that leak through despite stop-words
    filler = {'new','based','data','systems','system','research','project','high',
              'model','models','use','used','using','study','studies','methods',
              'method','approach','different','provide','provides','specific',
              'general','novel','result','results','number','across','within',
              'design','designs','well','also','one','two','three','make','made'}
    picks = [t for t in terms if not any(w in filler for w in t.split())][:3]
    if len(picks) < 2:
        picks = terms[:3]
    return ' · '.join(picks)

subtopics_raw['subtopic_label'] = subtopics_raw['top_terms'].map(auto_label)

cols = ['parent_topic','sub_id','subtopic_label','n_docs','total_dollars','top_terms','top_faculty','top_grants']
subtopics_final = subtopics_raw[cols].sort_values(['parent_topic','total_dollars'], ascending=[True,False])
subtopics_final.to_csv(OUTPUTS / 'subtopics.csv', index=False)
print(f'wrote {OUTPUTS / "subtopics.csv"}')

# Print a compact per-parent view
for parent, block in subtopics_final.groupby('parent_topic'):
    print(f'\n=== {parent} ===')
    for _, r in block.iterrows():
        print(f'  • {r.subtopic_label:55s} n={r.n_docs:3d}  ${r.total_dollars/1e6:5.1f}M')
        print(f'      terms  : {r.top_terms}')
        if r.top_faculty:
            print(f'      faculty: {r.top_faculty}')

## 3 · Topic dendrogram (Jensen–Shannon distance)

Which of the 8 parent topics are semantic cousins? We treat each topic's word distribution (row of `lda.components_`, normalised to a probability vector over the vocabulary) as a point, compute pairwise **Jensen–Shannon distance**, and cluster with average-linkage.

In [ ]:
from scipy.spatial.distance import jensenshannon
from scipy.cluster.hierarchy import linkage, dendrogram

# Normalise each topic's word distribution to a probability vector
P = lda.components_ / lda.components_.sum(axis=1, keepdims=True)

n_topics = P.shape[0]
D = np.zeros((n_topics, n_topics))
for i in range(n_topics):
    for j in range(i+1, n_topics):
        D[i, j] = D[j, i] = jensenshannon(P[i], P[j], base=2)

labels_for_dendro = [ALIGNED_LABELS[k] for k in range(n_topics)]

Z = linkage(squareform(D, checks=False), method='average')

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(Z, labels=labels_for_dendro, leaf_rotation=25, leaf_font_size=10,
           color_threshold=0.7*max(Z[:,2]), ax=ax)
ax.set_ylabel('Jensen–Shannon distance (average linkage)')
ax.set_title('Topic dendrogram — which of the 8 topics are cousins?')
plt.tight_layout()
plt.savefig(FIG_DIR / 'w7_topic_dendrogram.png', dpi=140, bbox_inches='tight')
plt.savefig(OUTPUTS / 'w7_topic_dendrogram.png', dpi=140, bbox_inches='tight')
plt.show()

# Print the distance matrix for the report
D_df = pd.DataFrame(D, index=labels_for_dendro, columns=labels_for_dendro).round(3)
print('\nJS distance matrix (0 = identical, ~1 = fully disjoint):')
print(D_df)

## 4 · UMAP projection of all grant abstracts

Reduce every abstract to a 2-D point via TF-IDF → UMAP so we can *see* the corpus. Two artefacts:

- **Static 2×2 panel** ([`outputs/w7_umap_grants.png`](../outputs/w7_umap_grants.png)) coloured four different ways (topic, college, agency, year).
- **Interactive Plotly HTML** ([`docs/07_grant_projection.html`](../docs/07_grant_projection.html)) — hover to see title, PIs, agency, $, year. This is the artefact meant for exploration.

In [ ]:
# TF-IDF over the cleaned corpus
tfidf = TfidfVectorizer(max_df=0.6, min_df=5, ngram_range=(1,2),
                        token_pattern=r'\b[a-z]{3,}\b', stop_words='english',
                        max_features=15000)
Xt = tfidf.fit_transform(abs_clean.clean)
print(f'TF-IDF matrix: {Xt.shape}')

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                    random_state=RNG, n_components=2)
emb = reducer.fit_transform(Xt)
print(f'UMAP embedding: {emb.shape}')

In [ ]:
# Assemble a plotting frame with all colour dimensions
plot_df = abs_clean[['id','sourceactivityid','title']].copy()
plot_df['grant_id'] = plot_df['sourceactivityid'].astype(str)
plot_df['x'] = emb[:, 0]
plot_df['y'] = emb[:, 1]
plot_df['topic_label'] = abs_clean_with_pred['topic_label'].values
plot_df['topic_prob']  = abs_clean_with_pred['topic_prob'].values

# Grant metadata
plot_df = plot_df.merge(grants[['grant_id','totaldollars','startdateyear','agencyname']],
                         on='grant_id', how='left')
plot_df['totaldollars']  = plot_df['totaldollars'].fillna(0)
plot_df['startdateyear'] = plot_df['startdateyear'].fillna(0).astype(int)

# Attach lead PI (first non-copi if any, else first) + college
lead = (faculty_grants.sort_values(['grant_id','is_copi'])
        .drop_duplicates('grant_id')[['grant_id','faculty_id','faculty_name']])
lead = lead.merge(faculty[['faculty_id','superior_academic_unit']], on='faculty_id', how='left')
lead = lead.rename(columns={'superior_academic_unit':'lead_college','faculty_name':'lead_pi'})
plot_df = plot_df.merge(lead[['grant_id','lead_pi','lead_college']], on='grant_id', how='left')
plot_df['lead_college'] = plot_df['lead_college'].astype(str).replace({'nan': 'Unknown'}).fillna('Unknown')
plot_df['lead_college_grp'] = plot_df.lead_college.where(plot_df.lead_college.isin(KEEP_COLLEGES), 'Other')

# Agency bucket
def agency_bucket(a):
    if not isinstance(a, str): return 'Unknown'
    al = a.lower()
    if 'national science foundation' in al: return 'NSF'
    if 'national institutes of health' in al: return 'NIH'
    if any(x in al for x in ['naval','army','air force','defense','darpa']): return 'DoD'
    if 'energy' in al: return 'DoE'
    if 'nasa' in al or 'aeronautics' in al: return 'NASA'
    return 'Other'
plot_df['agency_grp'] = plot_df.agencyname.map(agency_bucket).fillna('Unknown')

plot_df.head(3)

In [ ]:
# 2×2 static matplotlib panel
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

def scatter_by(ax, col, title, palette=None, legend=True, size=8):
    cats = plot_df[col].astype(str)
    order = cats.value_counts().index.tolist()
    if palette is None:
        pal = sns.color_palette('tab10', len(order))
    else:
        pal = palette
    for c, color in zip(order, pal):
        m = cats == c
        ax.scatter(plot_df.loc[m, 'x'], plot_df.loc[m, 'y'],
                   s=size, alpha=0.55, color=color, label=c, linewidths=0)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    if legend:
        ax.legend(fontsize=7, markerscale=1.5, loc='center left', bbox_to_anchor=(1.0, 0.5))

scatter_by(axes[0,0], 'topic_label', 'Coloured by LDA topic (k=8)')
scatter_by(axes[0,1], 'lead_college_grp', 'Coloured by lead PI college')
scatter_by(axes[1,0], 'agency_grp', 'Coloured by agency (bucketed)',
           palette=sns.color_palette('Set2', 8))

# Panel D: year gradient, sized by $
ax = axes[1,1]
years = plot_df.startdateyear.replace(0, np.nan)
sizes = np.log1p(plot_df.totaldollars) * 4
sc = ax.scatter(plot_df.x, plot_df.y, c=years, cmap='viridis', s=sizes, alpha=0.55, linewidths=0)
ax.set_title('Coloured by start year, sized by log($)')
ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label='Start year')

plt.suptitle('UMAP projection of grant abstracts (TF-IDF → cosine → UMAP)', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / 'w7_umap_grants.png', dpi=140, bbox_inches='tight')
plt.savefig(OUTPUTS / 'w7_umap_grants.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Interactive Plotly HTML — one figure with buttons to switch colour dimension
hover_cols = ['title','lead_pi','lead_college','agencyname','totaldollars','startdateyear','topic_label','topic_prob']
plot_df_html = plot_df.copy()
plot_df_html['title'] = plot_df_html['title'].astype(str).str[:120]
plot_df_html['totaldollars_fmt'] = plot_df_html['totaldollars'].map(lambda x: f'${x:,.0f}')

# We build 4 traces (one per colour dimension) and add a menu to toggle visibility.
import plotly.graph_objects as go

def make_traces(color_col, name):
    cats = sorted(plot_df_html[color_col].astype(str).unique())
    palette = px.colors.qualitative.Bold + px.colors.qualitative.Vivid
    traces = []
    for i, c in enumerate(cats):
        m = plot_df_html[color_col].astype(str) == c
        d = plot_df_html.loc[m]
        traces.append(go.Scattergl(
            x=d.x, y=d.y, mode='markers', name=c,
            legendgroup=name, legendgrouptitle_text=name, showlegend=True,
            marker=dict(size=6, opacity=0.6, color=palette[i % len(palette)]),
            customdata=np.stack([d.title, d.lead_pi.astype(str), d.lead_college.astype(str),
                                  d.agencyname.astype(str), d.totaldollars_fmt,
                                  d.startdateyear.astype(str), d.topic_label.astype(str),
                                  d.topic_prob.round(2).astype(str)], axis=-1),
            hovertemplate=(
                '<b>%{customdata[0]}</b><br>'
                'PI: %{customdata[1]} (%{customdata[2]})<br>'
                'Agency: %{customdata[3]}<br>'
                'Amount: %{customdata[4]}<br>'
                'Year: %{customdata[5]}<br>'
                'Topic: %{customdata[6]} (p=%{customdata[7]})<extra></extra>'
            ),
            visible=(name == 'Topic')
        ))
    return traces, len(cats)

fig = go.Figure()
group_ranges = {}
cursor = 0
for gname, gcol in [('Topic', 'topic_label'), ('College', 'lead_college_grp'),
                    ('Agency', 'agency_grp')]:
    tr, n = make_traces(gcol, gname)
    for t in tr: fig.add_trace(t)
    group_ranges[gname] = (cursor, cursor + n)
    cursor += n

total = cursor
def vis_for(gname):
    lo, hi = group_ranges[gname]
    v = [False] * total
    for i in range(lo, hi): v[i] = True
    return v

# Lock axes to the full data extent so toggling categories doesn't rescale
_xpad = (plot_df_html.x.max() - plot_df_html.x.min()) * 0.05
_ypad = (plot_df_html.y.max() - plot_df_html.y.min()) * 0.05
_xrange = [plot_df_html.x.min() - _xpad, plot_df_html.x.max() + _xpad]
_yrange = [plot_df_html.y.min() - _ypad, plot_df_html.y.max() + _ypad]

fig.update_layout(
    title='NEU grant abstracts — UMAP projection (hover for details)',
    width=1050, height=720,
    xaxis=dict(visible=False, range=_xrange, fixedrange=False,
               scaleanchor='y', scaleratio=1),
    yaxis=dict(visible=False, range=_yrange, fixedrange=False),
    margin=dict(l=40, r=260, t=90, b=40),
    legend=dict(x=1.02, y=1.0, xanchor='left', yanchor='top'),
    updatemenus=[dict(
        type='buttons', direction='right', x=0.5, y=1.08, xanchor='center',
        buttons=[dict(label=gname, method='update',
                       args=[{'visible': vis_for(gname)},
                             {'title': f'NEU grant abstracts — coloured by {gname}'}])
                  for gname in group_ranges]
    )]
)

out_html = DOCS / '07_grant_projection.html'
fig.write_html(out_html, include_plotlyjs='cdn', full_html=True)
print(f'wrote {out_html}')
print(f'points plotted: {len(plot_df_html):,}')

## 4b · UMAP via SPECTER2 embeddings (scientific-paper transformer)

Section 4 UMAP used **TF-IDF over abstract text** — fast, simple baseline.

Here we swap in **SPECTER2** ([`allenai/specter2_base`](https://huggingface.co/allenai/specter2_base) + proximity adapter), a transformer trained on 6M citation-linked
scientific-paper triplets. It emits a 768-dim vector per abstract that
encodes semantic relatedness ("papers that would cite each other should be
close"), a much richer similarity signal than word overlap.

Embeddings are pre-computed by [`src/build_specter2_embeddings.py`](../src/build_specter2_embeddings.py) and cached to `data/processed/specter2_embeddings.npy`
(one-time ~5 min on CPU, ~8 MB on disk). This cell just loads them and
re-runs UMAP, so re-executing the notebook is fast.

In [ ]:
# Load cached SPECTER2 embeddings (keyed on grant_id)
_spec_vecs = np.load(PROCESSED / 'specter2_embeddings.npy')
_spec_ids  = (PROCESSED / 'specter2_ids.txt').read_text().splitlines()
print(f'SPECTER2 cache: {_spec_vecs.shape}  ({len(_spec_ids)} ids)')

_id_to_row = {sid: i for i, sid in enumerate(_spec_ids)}
_keep_idx = [_id_to_row[i] for i in abs_clean['id'].astype(str) if i in _id_to_row]
_kept_mask = abs_clean['id'].astype(str).isin(_id_to_row)
Xs = _spec_vecs[_keep_idx]
print(f'SPECTER2 vectors for cleaned corpus: {Xs.shape} (of {len(abs_clean)} abs)')

reducer_spec = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                          random_state=RNG, n_components=2)
emb_spec = reducer_spec.fit_transform(Xs)
print(f'SPECTER2 UMAP embedding: {emb_spec.shape}')

In [ ]:
# 2x2 static panel for SPECTER2 layout
plot_df_spec = abs_clean[_kept_mask][['id','sourceactivityid','title']].reset_index(drop=True).copy()
plot_df_spec['grant_id'] = plot_df_spec['sourceactivityid'].astype(str)
plot_df_spec['x'] = emb_spec[:, 0]
plot_df_spec['y'] = emb_spec[:, 1]
plot_df_spec = plot_df_spec.merge(
    plot_df[['id','topic_label','topic_prob','lead_pi','lead_college','lead_college_grp',
             'agency_grp','agencyname','totaldollars','startdateyear']],
    on='id', how='left')
plot_df_spec['totaldollars']  = plot_df_spec['totaldollars'].fillna(0)
plot_df_spec['startdateyear'] = plot_df_spec['startdateyear'].fillna(0).astype(int)
plot_df_spec['lead_college_grp'] = plot_df_spec['lead_college_grp'].fillna('Other')
plot_df_spec['agency_grp']       = plot_df_spec['agency_grp'].fillna('Unknown')

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
def _scat(ax, df, col, title, palette=None, size=8):
    cats = df[col].astype(str)
    order = cats.value_counts().index.tolist()
    pal = palette or sns.color_palette('tab10', len(order))
    for c, color in zip(order, pal):
        m = cats == c
        ax.scatter(df.loc[m,'x'], df.loc[m,'y'], s=size, alpha=0.55, color=color, label=c, linewidths=0)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=7, markerscale=1.5, loc='center left', bbox_to_anchor=(1.0, 0.5))
_scat(axes[0,0], plot_df_spec, 'topic_label', 'Coloured by LDA topic (k=8)')
_scat(axes[0,1], plot_df_spec, 'lead_college_grp', 'Coloured by lead PI college')
_scat(axes[1,0], plot_df_spec, 'agency_grp', 'Coloured by agency (bucketed)', palette=sns.color_palette('Set2', 8))
ax = axes[1,1]
years = plot_df_spec.startdateyear.replace(0, np.nan)
sizes = np.log1p(plot_df_spec.totaldollars) * 4
sc = ax.scatter(plot_df_spec.x, plot_df_spec.y, c=years, cmap='viridis', s=sizes, alpha=0.55, linewidths=0)
ax.set_title('Coloured by start year, sized by log($)')
ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label='Start year')
plt.suptitle('UMAP projection via SPECTER2 (scientific-paper transformer → cosine → UMAP)', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / 'w7_umap_grants_specter2.png', dpi=140, bbox_inches='tight')
plt.savefig(OUTPUTS / 'w7_umap_grants_specter2.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Interactive Plotly HTML for SPECTER2 projection
import plotly.graph_objects as go
_pd = plot_df_spec.copy()
_pd['title'] = _pd['title'].astype(str).str[:120]
_pd['totaldollars_fmt'] = _pd['totaldollars'].map(lambda x: f'${x:,.0f}')

def make_traces_spec(color_col, name):
    cats = sorted(_pd[color_col].astype(str).unique())
    palette = px.colors.qualitative.Bold + px.colors.qualitative.Vivid
    traces = []
    for i, c in enumerate(cats):
        m = _pd[color_col].astype(str) == c
        d = _pd.loc[m]
        traces.append(go.Scattergl(
            x=d.x, y=d.y, mode='markers', name=c,
            legendgroup=name, legendgrouptitle_text=name, showlegend=True,
            marker=dict(size=6, opacity=0.6, color=palette[i % len(palette)]),
            customdata=np.stack([d.title, d.lead_pi.astype(str), d.lead_college.astype(str),
                                  d.agencyname.astype(str), d.totaldollars_fmt,
                                  d.startdateyear.astype(str), d.topic_label.astype(str),
                                  d.topic_prob.round(2).astype(str)], axis=-1),
            hovertemplate=('<b>%{customdata[0]}</b><br>PI: %{customdata[1]} (%{customdata[2]})<br>'
                            'Agency: %{customdata[3]}<br>Amount: %{customdata[4]}<br>'
                            'Year: %{customdata[5]}<br>Topic: %{customdata[6]} (p=%{customdata[7]})<extra></extra>'),
            visible=(name == 'Topic')))
    return traces, len(cats)

fig = go.Figure()
group_ranges = {}; cursor = 0
for gname, gcol in [('Topic','topic_label'), ('College','lead_college_grp'), ('Agency','agency_grp')]:
    tr, n = make_traces_spec(gcol, gname)
    for t in tr: fig.add_trace(t)
    group_ranges[gname] = (cursor, cursor + n); cursor += n
total = cursor
def _vis(g):
    lo, hi = group_ranges[g]; return [lo <= i < hi for i in range(total)]

# Lock axes to the full data extent so toggling categories doesn't rescale
_xpad = (_pd.x.max() - _pd.x.min()) * 0.05
_ypad = (_pd.y.max() - _pd.y.min()) * 0.05
_xrange = [_pd.x.min() - _xpad, _pd.x.max() + _xpad]
_yrange = [_pd.y.min() - _ypad, _pd.y.max() + _ypad]

fig.update_layout(
    title='NEU grant abstracts — SPECTER2 UMAP projection (hover for details)',
    width=1050, height=720,
    xaxis=dict(visible=False, range=_xrange,
               scaleanchor='y', scaleratio=1),
    yaxis=dict(visible=False, range=_yrange),
    margin=dict(l=40, r=260, t=90, b=40),
    legend=dict(x=1.02, y=1.0, xanchor='left', yanchor='top'),
    updatemenus=[dict(type='buttons', direction='right', x=0.5, y=1.08, xanchor='center',
        buttons=[dict(label=g, method='update',
                      args=[{'visible': _vis(g)},
                            {'title': f'NEU grant abstracts (SPECTER2) — coloured by {g}'}])
                 for g in group_ranges])])

out_html = DOCS / '07_grant_projection_specter2.html'
fig.write_html(out_html, include_plotlyjs='cdn', full_html=True)
print(f'wrote {out_html}')
print(f'points plotted: {len(_pd):,}')

### TF-IDF vs SPECTER2 — how to read the two layouts

Both layouts use identical UMAP hyperparameters, so differences come from the
input vector, not from UMAP tuning.

- **TF-IDF** ([w7_umap_grants.png](../outputs/w7_umap_grants.png)) — treats
  each unigram/bigram as an independent signal.
- **SPECTER2** ([w7_umap_grants_specter2.png](../outputs/w7_umap_grants_specter2.png))
  — captures semantic similarity (synonyms, paraphrases, cross-vocabulary
  concepts). Requires the ~500 MB model download + ~5 min encoding, cached
  after that.

If the same LDA topic points form tighter, more separated blobs in the
SPECTER2 layout than in the TF-IDF layout, that's evidence the LDA topics are
capturing real semantic structure.

## Deliverables

- [`outputs/w7_topic_by_college_rownorm.png`](../outputs/w7_topic_by_college_rownorm.png) — row-normalised topic × college heatmap
- [`outputs/college_profiles.csv`](../outputs/college_profiles.csv) + [`docs/college_profiles.html`](../docs/college_profiles.html) — per-college profile cards
- [`outputs/subtopics.csv`](../outputs/subtopics.csv) — 32 sub-topics (8 parents × 4) with top terms, top faculty, top grants
- [`outputs/w7_topic_dendrogram.png`](../outputs/w7_topic_dendrogram.png) — JS-distance dendrogram over the 8 parent topics
- [`outputs/w7_umap_grants.png`](../outputs/w7_umap_grants.png) — static 2×2 UMAP panel
- [`docs/07_grant_projection.html`](../docs/07_grant_projection.html) — interactive Plotly projection with a topic / college / agency toggle
- [`outputs/w7_umap_grants_specter2.png`](../outputs/w7_umap_grants_specter2.png) — SPECTER2 UMAP panel
- [`docs/07_grant_projection_specter2.html`](../docs/07_grant_projection_specter2.html) — SPECTER2 interactive projection
